Nhóm các pixel có cùng màu - K-mean

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import os

In [2]:
# --- 0. CÀI ĐẶT CÁC ĐƯỜNG DẪN ---
INPUT_IMAGE_PATH = 'data/dog_cat.jpg'
OUTPUT_DIR = 'output'

# Tạo thư mục 'output' nếu nó chưa tồn tại
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    
print(f"Sẽ lưu kết quả vào thư mục: {OUTPUT_DIR}")

# --- 1. TẢI VÀ CHUẨN BỊ DỮ LIỆU ---
print(f"Đang tải ảnh từ: {INPUT_IMAGE_PATH}")
img = cv2.imread(INPUT_IMAGE_PATH)

if img is None:
    print(f"LỖI: Không thể tìm thấy ảnh tại '{INPUT_IMAGE_PATH}'.")
    print("Vui lòng đảm bảo bạn có ảnh 'dog_cat.jpg' trong thư mục 'data'.")
else:
    # Chuyển đổi sang ảnh xám
    gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    
    # Resize ảnh (W=200, H=140)
    gray_img = cv2.resize(gray_img, (200, 140))
    print(f"Kích thước ảnh đã resize: {gray_img.shape}")
    
    # Tính dung lượng ảnh gốc (chưa nén, trong RAM)
    original_size_bytes = gray_img.nbytes
    print(f"Dung lượng ảnh gốc (trong RAM): {original_size_bytes} bytes")

    # --- Hiển thị và lưu ảnh gốc ---
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(gray_img, cmap='gray')
    plt.title('Ảnh xám gốc (Resized)')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.hist(gray_img.ravel(), bins=256, color='gray', alpha=0.7)
    plt.title('Histogram ảnh gốc')
    plt.xlabel('Cường độ Pixel')
    plt.ylabel('Tần suất')
    plt.tight_layout()
    
    # Lưu biểu đồ ảnh gốc
    output_path = os.path.join(OUTPUT_DIR, '00_original_image.png')
    plt.savefig(output_path)
    print(f"Đã lưu ảnh gốc và histogram vào: {output_path}")
    plt.close() # Đóng biểu đồ để tiết kiệm bộ nhớ

    # --- 2. CHUẨN BỊ K-MEANS ---
    # Reshape từ (140, 200) -> (28000, 1)
    X_gray = gray_img.reshape(-1, 1)
    print(f"Kích thước dữ liệu K-Means (Pixels, Features): {X_gray.shape}")


    # --- 3. CHẠY K-MEANS VÀ LƯU KẾT QUẢ ---
    print("\n--- Bắt đầu vòng lặp K-Means ---")

    for K in [5, 10, 15, 20]:
        print(f"\nĐang xử lý với K = {K}...")
        
        # Huấn luyện mô hình K-Means
        kmeans = KMeans(n_clusters=K, random_state=0, n_init=10).fit(X_gray)
        
        # Lấy nhãn và tâm cụm
        label = kmeans.labels_
        centers = kmeans.cluster_centers_

        # --- TÍNH TOÁN DUNG LƯỢNG NÉN ---
        size_palette_bytes = centers.nbytes
        size_labels_bytes = label.astype(np.uint8).nbytes
        compressed_size_bytes = size_palette_bytes + size_labels_bytes
        
        print(f"  Dung lượng Bảng màu (K={K}): {size_palette_bytes} bytes")
        print(f"  Dung lượng Bản đồ nhãn: {size_labels_bytes} bytes")
        print(f"  TỔNG DUNG LƯỢNG NÉN (K={K}): {compressed_size_bytes} bytes")
        print(f"  So với gốc ({original_size_bytes} bytes), nén còn: {compressed_size_bytes / original_size_bytes:.2%}")

        # --- TÁI TẠO ẢNH ---
        # Gán giá trị của tâm cụm cho tất cả các pixel thuộc cụm đó
        img_reconstructed_flat = centers[label]
        img_reconstructed = img_reconstructed_flat.reshape(gray_img.shape)

        # --- HIỂN THỊ VÀ LƯU KẾT QUẢ ---
        fig = plt.figure(figsize=(10, 5))

        # Ảnh K-Means
        plt.subplot(1, 2, 1)
        plt.title(f"Ảnh K = {K}\nDung lượng nén: {compressed_size_bytes} bytes")
        plt.imshow(img_reconstructed, interpolation='nearest', cmap='gray')
        plt.axis('off')

        # Histogram của ảnh K-Means
        plt.subplot(1, 2, 2)
        plt.hist(img_reconstructed.ravel(), bins=256, color='gray', alpha=0.7)
        plt.title(f'Histogram cho K = {K}')
        plt.xlabel('Cường độ Pixel')
        plt.ylabel('Tần suất')

        plt.tight_layout()
        
        # Lưu file kết quả vào thư mục output
        output_path = os.path.join(OUTPUT_DIR, f'kmeans_K{K}_result.png')
        plt.savefig(output_path)
        print(f"Đã lưu kết quả K={K} vào: {output_path}")
        plt.close(fig) # Đóng biểu đồ hiện tại

    print("\n--- HOÀN THÀNH TẤT CẢ ---")

Sẽ lưu kết quả vào thư mục: output
Đang tải ảnh từ: data/dog_cat.jpg
Kích thước ảnh đã resize: (140, 200)
Dung lượng ảnh gốc (trong RAM): 28000 bytes
Đã lưu ảnh gốc và histogram vào: output\00_original_image.png
Kích thước dữ liệu K-Means (Pixels, Features): (28000, 1)

--- Bắt đầu vòng lặp K-Means ---

Đang xử lý với K = 5...
  Dung lượng Bảng màu (K=5): 40 bytes
  Dung lượng Bản đồ nhãn: 28000 bytes
  TỔNG DUNG LƯỢNG NÉN (K=5): 28040 bytes
  So với gốc (28000 bytes), nén còn: 100.14%
Đã lưu kết quả K=5 vào: output\kmeans_K5_result.png

Đang xử lý với K = 10...
  Dung lượng Bảng màu (K=10): 80 bytes
  Dung lượng Bản đồ nhãn: 28000 bytes
  TỔNG DUNG LƯỢNG NÉN (K=10): 28080 bytes
  So với gốc (28000 bytes), nén còn: 100.29%
Đã lưu kết quả K=10 vào: output\kmeans_K10_result.png

Đang xử lý với K = 15...
  Dung lượng Bảng màu (K=15): 120 bytes
  Dung lượng Bản đồ nhãn: 28000 bytes
  TỔNG DUNG LƯỢNG NÉN (K=15): 28120 bytes
  So với gốc (28000 bytes), nén còn: 100.43%
Đã lưu kết quả K=15 và